# Location Dimension Loader

Maintains the `warehouse.dim_location` dimension table with incremental refresh capability.

## Purpose
Track job posting locations with geographic hierarchy for location-based analytics.

## Key Features
* Stable surrogate keys for locations (auto-increment)
* Geographic hierarchy (country, state/province, city)
* Remote work type classification
* Location parsing from normalized strings
* SCD Type 1 (overwrite on change)
* Idempotent: safe to re-run

## Architecture
**Source**: Silver layer current jobs (`workspace.silver.silver_jobs_current`)  
**Target**: `workspace.warehouse.dim_location`  
**Metadata**: `workspace.metadata.dim_location_refresh_log`  
**Mode**: Incremental (merge new/updated locations)

## Batch Processing
* Tracks refresh history in metadata table
* Auto-assigns surrogate keys to new locations
* Updates existing locations with latest data
* Validates geographic distribution after each refresh

In [0]:
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")

FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType, LongType
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"
SILVER_SCHEMA = f"{CATALOG}.silver"

SOURCE_TABLE = f"{SILVER_SCHEMA}.silver_jobs_current"
TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_location"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_location_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

print(f"Run ID: {run_id}")
print(f"Source table: {SOURCE_TABLE}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")

In [0]:
%sql
-- Create location dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_location (
  location_sk BIGINT NOT NULL COMMENT 'Surrogate key for location',
  location_name STRING NOT NULL COMMENT 'Full normalized location string',
  city STRING COMMENT 'City name',
  state_province STRING COMMENT 'State or province',
  country STRING NOT NULL COMMENT 'Country name',
  remote_type STRING COMMENT 'Remote work type classification',
  is_remote BOOLEAN NOT NULL COMMENT 'Is remote location',
  is_active BOOLEAN NOT NULL COMMENT 'Is location active',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
  CONSTRAINT pk_dim_location PRIMARY KEY (location_sk)
)
USING DELTA
COMMENT 'Location dimension with geographic hierarchy';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_location_refresh_log (
  run_id STRING,
  locations_extracted INT,
  locations_inserted INT,
  locations_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks location dimension refresh history';

In [0]:
print("Extracting distinct locations from silver layer...", end=" ")

# Load distinct locations from silver layer
jobs_df = spark.table(SOURCE_TABLE)

# Extract and parse location components
location_extract_df = jobs_df.filter(F.col("location_norm").isNotNull()).select(
    F.col("location_norm"),
    F.col("remote_type")
).distinct()

# Parse location components (city, state, country)
location_parsed_df = location_extract_df.select(
    F.col("location_norm").alias("location_name"),
    F.col("remote_type"),
    # Parse city (first component)
    F.when(
        F.col("location_norm").contains(","),
        F.trim(F.split(F.col("location_norm"), ",")[0])
    ).otherwise(F.col("location_norm")).alias("city"),
    # Parse state/province (middle component for 3-part, second for 2-part)
    F.when(
        F.size(F.split(F.col("location_norm"), ",")) >= 3,
        F.trim(F.split(F.col("location_norm"), ",")[1])
    ).when(
        F.size(F.split(F.col("location_norm"), ",")) == 2,
        F.trim(F.split(F.col("location_norm"), ",")[0])
    ).otherwise(None).alias("state_province"),
    # Parse country (last component if comma-separated, else 'Unknown')
    F.when(
        F.size(F.split(F.col("location_norm"), ",")) >= 3,
        F.trim(F.split(F.col("location_norm"), ",")[2])
    ).when(
        F.size(F.split(F.col("location_norm"), ",")) == 2,
        F.trim(F.split(F.col("location_norm"), ",")[1])
    ).otherwise("Unknown").alias("country"),
    # Remote flag
    F.when(F.col("remote_type") == "REMOTE", True).otherwise(False).alias("is_remote"),
    F.lit(True).alias("is_active")
)

locations_count = location_parsed_df.count()
print(f"✓ Extracted {locations_count} distinct locations")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(location_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:
# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("locations_extracted", IntegerType(), True),
    StructField("locations_inserted", IntegerType(), True),
    StructField("locations_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Processing locations into {TARGET_TABLE}...", end=" ")
    
    # Check if table schema matches expected schema FIRST
    existing_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    expected_cols = ['location_sk', 'location_name', 'city', 'state_province', 'country', 'remote_type', 'is_remote', 'is_active', 'updated_at']
    schema_matches = set(existing_cols) == set(expected_cols)
    
    # Only query existing locations if schema matches
    if schema_matches:
        existing_locations = spark.sql(f"SELECT location_name, location_sk FROM {TARGET_TABLE}")
        
        # Join to assign keys (existing or new)
        from pyspark.sql.window import Window
        
        locations_with_keys = location_parsed_df.alias("l").join(
            existing_locations.alias("e"),
            F.col("l.location_name") == F.col("e.location_name"),
            "left"
        )
        
        # Assign surrogate keys
        window_spec = Window.orderBy("l.location_name")
        
        locations_final = locations_with_keys.withColumn(
            "location_sk",
            F.coalesce(F.col("e.location_sk"), F.lit(max_sk) + F.row_number().over(window_spec))
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            F.col("location_sk").cast(LongType()),
            F.col("l.location_name").alias("location_name"),
            F.col("l.city").alias("city"),
            F.col("l.state_province").alias("state_province"),
            F.col("l.country").alias("country"),
            F.col("l.remote_type").alias("remote_type"),
            F.col("l.is_remote").alias("is_remote"),
            F.col("l.is_active").alias("is_active"),
            "updated_at"
        )
    else:
        # Schema mismatch: assign keys without joining to existing table
        from pyspark.sql.window import Window
        window_spec = Window.orderBy("location_name")
        
        locations_final = location_parsed_df.withColumn(
            "location_sk",
            (F.lit(max_sk) + F.row_number().over(window_spec)).cast(LongType())
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            "location_sk",
            "location_name",
            "city",
            "state_province",
            "country",
            "remote_type",
            "is_remote",
            "is_active",
            "updated_at"
        )
    
    # Create temp view for merge
    locations_final.createOrReplaceTempView("locations_to_merge")
    
    if FORCE_FULL_REFRESH or not schema_matches:
        # Full refresh: drop and recreate with new schema
        if not schema_matches:
            print(f"Schema mismatch detected. Performing full refresh...")
            spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
            spark.sql(f"""
            CREATE TABLE {TARGET_TABLE} (
              location_sk BIGINT NOT NULL COMMENT 'Surrogate key for location',
              location_name STRING NOT NULL COMMENT 'Full normalized location string',
              city STRING COMMENT 'City name',
              state_province STRING COMMENT 'State or province',
              country STRING NOT NULL COMMENT 'Country name',
              remote_type STRING COMMENT 'Remote work type classification',
              is_remote BOOLEAN NOT NULL COMMENT 'Is remote location',
              is_active BOOLEAN NOT NULL COMMENT 'Is location active',
              updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
              CONSTRAINT pk_dim_location PRIMARY KEY (location_sk)
            )
            USING DELTA
            COMMENT 'Location dimension with geographic hierarchy'
            """)
        else:
            spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        
        locations_final.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        locations_inserted = locations_final.count()
        locations_updated = 0
        print(f"✓ Full refresh: {locations_inserted} locations inserted")
    else:
        # Incremental: merge
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING locations_to_merge source
        ON target.location_name = source.location_name
        WHEN MATCHED THEN UPDATE SET
            target.city = source.city,
            target.state_province = source.state_province,
            target.country = source.country,
            target.remote_type = source.remote_type,
            target.is_remote = source.is_remote,
            target.is_active = source.is_active,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        locations_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM locations_to_merge
            WHERE location_name NOT IN (SELECT location_name FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        
        locations_updated = locations_count - locations_inserted
        
        print(f"✓ Merge complete: {locations_inserted} new, {locations_updated} updated")
    
    # Log to metadata
    metadata_data = [(
        run_id,
        locations_count,
        locations_inserted,
        locations_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "locations_extracted": locations_count,
        "locations_inserted": locations_inserted,
        "locations_updated": locations_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata
    metadata_data = [(
        run_id,
        locations_count if 'locations_count' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate location dimension
SELECT 
  COUNT(*) as total_locations,
  COUNT(DISTINCT country) as countries,
  COUNT(DISTINCT state_province) as states,
  COUNT(DISTINCT city) as cities,
  SUM(CASE WHEN is_remote THEN 1 ELSE 0 END) as remote_locations,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_locations
FROM workspace.warehouse.dim_location;

In [0]:
%sql
-- Sample locations by country and state
SELECT 
  location_sk,
  location_name,
  city,
  state_province,
  country,
  remote_type,
  is_remote,
  is_active,
  updated_at
FROM workspace.warehouse.dim_location
ORDER BY country, state_province, city
LIMIT 25;

In [0]:
%sql
-- Show refresh history
SELECT 
  run_id,
  locations_extracted,
  locations_inserted,
  locations_updated,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_location_refresh_log
ORDER BY processed_at DESC
LIMIT 10;